# 1주차 예제 — 따릉이 EDA/시각화 + KRX API 실습

이 노트북은 세 파트로 구성됩니다.

**결측치(missing value)**는 기록되어야 할 값이 비어 있는 상태이고, **이상치(outlier)**는 같은 변수의 다른 관측값과 비교해 유난히 멀리 떨어진 값입니다. 이상치는 입력 오류일 수도 있고 실제로 드문 정상 사례일 수도 있습니다.

1. **파이썬 · 판다스 맛보기** — 아주 작은 표로 기본 문법과 오늘 쓸 핵심 개념(결측치, 이상치)을 먼저 익힙니다. 판다스가 처음이라면 이 파트부터 실행합니다.
2. **따릉이 대여이력 EDA** — 실제 데이터로 결측치/이상치 확인, 기술통계, 다양한 시각화
3. **KRX API 실습** — 승인된 키를 코드에 노출하지 않고 KRX 지수·코스닥 종목을 직접 수집해 JSON 검증, CSV 캐시, EDA까지 진행합니다.

노션 자료(`notion.md`)의 단계 번호·용어집과 이 노트북의 소제목이 그대로 맞물립니다. 노션을 먼저 읽고 여기서 직접 실행한 뒤 결과를 비교합니다.


## Part 1. 파이썬 · 판다스 맛보기

`pandas`는 표(엑셀 같은 행/열 구조) 형태의 데이터를 다루는 라이브러리입니다. 핵심 자료구조는 두 가지입니다.

- `DataFrame` — 표 전체 (행 × 열)
- `Series` — 표의 한 열

5행으로 구성된 작은 예제 표를 만들어 기본 문법을 하나씩 차근차근 확인합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Windows에서 한글이 깨지지 않도록 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


### 1-1. DataFrame 만들기


In [ ]:
toy = pd.DataFrame({
    '요일': ['월', '화', '수', '목', '금'],
    '대여건수': [320, 280, 350, 300, 410],
    '평균기온': [1.2, 0.5, 2.1, 3.0, 4.5],
})
toy


### 1-2. 열 선택하기

`df['열이름']`으로 한 열(Series)을 꺼내고, `df[['열1', '열2']]`로 여러 열을 동시에 꺼낼 수 있습니다.


In [ ]:
toy['대여건수']


In [ ]:
toy[['요일', '대여건수']]


### 1-3. 조건으로 걸러내기

`df[조건]` 형태로 조건을 만족하는 행만 골라낼 수 있습니다. 데이터 분석에서 가장 많이 쓰는 패턴 중 하나입니다.


In [ ]:
toy[toy['대여건수'] > 300]


### 1-4. 정렬하기


In [ ]:
toy.sort_values('대여건수', ascending=False)


### 1-5. 요약 통계

`mean()`, `max()`, `min()`처럼 열 하나에 바로 통계 함수를 적용할 수 있고, `describe()`는 여러 통계를 한 번에 보여줍니다.


In [ ]:
print('평균 대여건수:', toy['대여건수'].mean())
print('최대 대여건수:', toy['대여건수'].max())
toy['대여건수'].describe()


### 1-6. 그룹별 집계 — `groupby`

실제 데이터에서는 '평일 평균은 얼마입니까?', '요일별 합계는 얼마입니까?'와 같은 질문을 자주 만납니다. 이럴 때 `groupby`를 사용합니다. 여기서는 월요일부터 금요일까지를 '평일'이라는 새 열로 구분한 뒤 그룹 평균을 계산합니다.


In [ ]:
toy['구분'] = ['평일', '평일', '평일', '평일', '평일']
toy.groupby('구분')['대여건수'].mean()


### 1-7. 그래프 그리기

`matplotlib`으로 시각화합니다. `fig, ax = plt.subplots()`로 그림판을 만들고, `ax.bar()`, `ax.plot()` 같은 함수로 그립니다.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(toy['요일'], toy['대여건수'])
ax.set_title('요일별 대여건수 (간단한 예제)')
plt.show()


### 1-8. 결측치란 무엇인가 (간단한 예제)

**결측치(missing value)**는 원래 관측하거나 입력해야 할 값이 기록되지 않은 상태입니다. pandas에서는 `None`이나 `NaN` 등을 결측치로 인식합니다. `0`은 값이 0이라고 기록된 것이므로 결측치가 아니며, 빈 문자열이나 `-`는 데이터에 따라 결측을 뜻하더라도 자동으로 인식되지 않을 수 있습니다. 5명의 응답으로 만든 간단한 예제 설문 데이터로 확인합니다.


In [ ]:
survey = pd.DataFrame({
    '이름': ['A', 'B', 'C', 'D', 'E'],
    '나이': [25, 31, None, 40, 29],       # 응답을 깜빡 잊고 안 적음
    '소득': [3000, None, 4200, None, 3500], # 소득이 아주 높은 사람이 일부러 답을 안 함
})
survey


In [ ]:
survey.isna()  # True인 칸이 결측치


In [ ]:
missing_summary = pd.DataFrame({
    '결측 개수': survey.isna().sum(),
    '결측 비율': survey.isna().mean(),
})
missing_summary


**결측치는 발생 원인이 서로 다릅니다.** 이 간단한 예제에는 두 가지 상황을 가정했습니다.

- `나이`의 결측: 응답을 우연히 빠뜨렸고 다른 정보나 실제 나이와 무관하다고 가정하면 무작위 결측에 가깝습니다.
- `소득`의 결측: 소득이 아주 높은 사람일수록 답을 피하는 상황이라면 결측 여부 자체가 정보입니다. 이유를 확인하지 않고 평균으로 채우면 '고소득자일 가능성'이라는 신호를 지울 수 있습니다.

이 표만으로 실제 결측 원인을 확정하기는 어렵습니다. 먼저 개수와 비율을 확인하고, **'어떤 집단에서 어떤 수집 과정 때문에 없을까'라는 가설**을 세운 뒤 설문 방식이나 수집 기록을 함께 살펴보면 원인을 더 잘 이해할 수 있습니다. Part 2에서 같은 판단 순서를 실제 데이터에 적용합니다.


### 1-9. 이상치란 무엇인가 (간단한 예제) — IQR 방법

**이상치(outlier)**는 같은 변수의 다른 관측값과 비교해 유난히 멀리 떨어진 값입니다. 단위 오입력 같은 오류일 수도 있고, 실제로 드문 정상 사례일 수도 있습니다. 따라서 통계 규칙은 삭제 대상을 결정하는 판정선이 아니라 원본과 업무 맥락을 확인할 후보를 찾는 데 사용합니다. 자주 쓰는 규칙 중 하나가 **IQR(사분위범위) 방법**입니다.

1. Q1(25% 지점)과 Q3(75% 지점)을 구합니다.
2. `IQR = Q3 − Q1`로 계산합니다.
3. `Q1 − 1.5×IQR`보다 작거나 `Q3 + 1.5×IQR`보다 큰 값을 검토 후보로 표시합니다.


In [ ]:
rental_counts = pd.Series([10, 12, 11, 13, 12, 14, 11, 90])  # 90이 유독 커 보입니다

q1 = rental_counts.quantile(0.25)
q3 = rental_counts.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'IQR 탐색 경계: {lower} ~ {upper}')
candidate_mask = (rental_counts < lower) | (rental_counts > upper)
print('IQR 경계 밖 후보:', rental_counts[candidate_mask].tolist())


90은 IQR 탐색 경계 밖의 **검토 후보**로 표시됩니다. 이것만으로 90이 오류라고 결론 내릴 수는 없습니다. **IQR 방법은 평균이 아니라 분위수를 기준으로 삼기 때문에 극단값 한두 개가 기준 자체를 크게 흔들지 않는다**는 장점이 있습니다. Part 2에서 실제 이용시간·이용거리 데이터에 같은 논리를 적용합니다.


여기까지 pandas 기본기와 결측치·이상치 개념을 살펴봤습니다. 이제 실제 데이터에 같은 원리를 적용합니다. **이후의 코드는 앞에서 익힌 패턴인 열 선택 → 조건 필터 → 정렬·집계 → 시각화와 결측치·이상치 점검을 차례로 확장합니다.**


## Part 2. 따릉이 대여이력 EDA

서울시 공공자전거 따릉이의 2026년 2월 개별 대여 기록입니다. 대여시각, 이용시간, 이용거리, 이용자의 생년·성별 등이 담겨 있어 결측치·이상치 연습에 좋습니다. 원본은 [서울 열린데이터광장 — 서울특별시 공공자전거 대여이력 정보](https://data.seoul.go.kr/dataList/OA-15182/A/1/datasetView.do)에서 내려받을 수 있습니다. 이 노트북은 `dataset/extracted/따릉이 공공데이터/03_대여이력/서울특별시 공공자전거 대여이력 정보_2602.csv`에 준비된 파일을 읽습니다. 아래 소제목 번호는 `notion.md`의 '1단계 ~ 6단계'와 그대로 대응합니다.


### 1단계 — 데이터 불러오기 & 구조 파악

**질문**: 이 데이터는 몇 행, 몇 열이며 어떤 자료형의 값이 들어 있습니까?

원본 파일은 약 180만 행이므로 필요한 컬럼만 `usecols`로 골라 불러옵니다. 이 실습 파일의 인코딩은 `cp949`이며, `utf-8`로 읽으면 오류가 발생합니다. 공공데이터의 인코딩은 파일마다 다를 수 있으므로 다른 파일을 사용할 때는 배포 안내나 실제 파일을 확인합니다.


In [ ]:
cols = ['대여일시', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류']
df = pd.read_csv(
    '../dataset/extracted/따릉이 공공데이터/03_대여이력/서울특별시 공공자전거 대여이력 정보_2602.csv',
    encoding='cp949',
    usecols=cols,
)
df.shape


In [ ]:
df.dtypes


In [ ]:
df.head()


### 2단계 — 결측치 확인

**질문**: 어떤 컬럼이 비어 있으며, 결측은 무작위로 나타납니까, 아니면 특정 그룹에 모여 있습니까? Part 1-8에서 익힌 질문을 같은 순서로 적용합니다.


In [ ]:
df.isna().mean()


In [ ]:
# 이용자종류별로 성별 결측 비율이 다른지 확인
df.groupby('이용자종류')['성별'].apply(lambda s: s.isna().mean())


→ 이용자종류별 결측률이 크게 다르면 가입·이용 절차에 따른 구조적 결측일 가능성을 시사합니다. 그러나 이 결과만으로 원인을 확정할 수는 없으므로 데이터 명세와 실제 수집 절차를 함께 확인합니다.


### 3단계 — 이상치 확인

**질문**: 분포에서 유난히 멀거나 업무 규칙을 벗어나는 값은 얼마나 섞여 있습니까? `0`은 결측치가 아니라 실제 기록된 값이므로, 0이 가능한 값인지 변수 정의와 수집 과정을 따로 확인합니다. Part 1-9에서 배운 IQR 방법으로 검토 후보도 표시합니다.


In [ ]:
df[['이용시간(분)', '이용거리(M)']].describe()


In [ ]:
q1 = df['이용시간(분)'].quantile(0.25)
q3 = df['이용시간(분)'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f'이용시간 IQR 탐색 상한: {upper:.1f}분')
print('IQR 상한 밖 후보 비율:', (df['이용시간(분)'] > upper).mean())


In [ ]:
print('이용시간 0분 비율:', (df['이용시간(분)'] == 0).mean())
print('이용거리 0M 비율:', (df['이용거리(M)'] == 0).mean())


### 4단계 — 파생변수 만들기

**질문**: 시간대·요일·나이로 나누어 보면 어떤 특징이 보입니까?

`대여일시`에서 날짜·시간대·요일을 뽑고, `생년`으로 나이를 계산합니다. 원본에 없던 컬럼을 새로 만드는 것을 **파생변수**라고 부릅니다.


In [ ]:
dt = pd.to_datetime(df['대여일시'])
df['날짜'] = dt.dt.date
df['연도'] = dt.dt.year
df['시간대'] = dt.dt.hour
df['요일'] = dt.dt.day_name()
df['나이'] = df['연도'] - df['생년']
df[['날짜', '시간대', '요일', '나이']].head()


### 5단계 — 파생변수도 다시 이상치 점검

**질문**: 방금 계산한 `나이`는 분석에 사용할 만큼 타당합니까?

계산으로 만든 컬럼도 값의 범위를 다시 검증하는 과정이 필요합니다. `describe()`로 최솟값과 최댓값을 살펴보고, `나이`처럼 0보다 작거나 현실적으로 가능한 최댓값을 크게 넘는 값이 섞여 있는지 확인합니다.


In [ ]:
df['나이'].describe()


In [ ]:
# 음수 나이, 100살이 넘는 나이 → 원본 '생년' 입력 오류로 추정
df[(df['나이'] < 0) | (df['나이'] > 100)][['생년', '연도', '나이']].head(10)


### 6단계 — 시각화로 질문에 답하기

지금부터는 Part 1에서 익힌 선택 → 집계 → 시각화 흐름을 반복하면서, 질문에 맞는 pandas와 matplotlib 함수를 하나씩 사용합니다.


**그래프 ① 막대그래프 — 이용자 구성**

질문: 회원 구성은 어떻게 이루어져 있습니까? → `value_counts()`로 개수를 센 뒤 막대그래프로 나타냅니다.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df['이용자종류'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('이용자종류별 대여건수')
ax.set_ylabel('대여건수')
plt.show()


**그래프 ② 막대그래프 — 시간대별 이용 패턴**

질문: 출퇴근 시간대에 대여가 모입니까? → `groupby().size()`로 시간대별 개수를 센 뒤 막대그래프로 나타냅니다.


In [ ]:
hourly = df.groupby('시간대').size()
fig, ax = plt.subplots(figsize=(8, 4))
hourly.plot(kind='bar', ax=ax)
ax.set_xlabel('시간대(시)')
ax.set_ylabel('대여건수')
ax.set_title('시간대별 대여건수')
plt.show()


**그래프 ③ 꺾은선그래프 — 날짜별 추이**

질문: 한 달 동안 이용량은 어떻게 변합니까? → 시간 흐름에 따른 변화는 꺾은선그래프로 살펴봅니다.


In [ ]:
daily = df.groupby('날짜').size()
fig, ax = plt.subplots(figsize=(9, 4))
daily.plot(kind='line', marker='o', ax=ax)
ax.set_xlabel('날짜')
ax.set_ylabel('대여건수')
ax.set_title('날짜별 대여건수 추이')
plt.xticks(rotation=45)
plt.show()


**그래프 ④ 히트맵 — 요일 × 시간대**

질문: 평일과 주말의 시간대 패턴은 다르게 나타납니까? → 두 범주인 요일과 시간대를 함께 보기 위해 `pivot_table`로 표를 만들고 히트맵으로 나타냅니다.


In [ ]:
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df.pivot_table(index='요일', columns='시간대', values='대여일시', aggfunc='count')
pivot = pivot.reindex(order)
pivot.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_xlabel('시간대(시)')
ax.set_title('요일 x 시간대 대여건수 히트맵')
fig.colorbar(im, ax=ax, label='대여건수')
plt.show()


**그래프 ⑤ Boxplot — 이용시간 이상치**

질문: IQR 규칙으로 표시되는 후보는 분포의 중심에서 얼마나 멀리 떨어져 있습니까? → boxplot에서 상자의 아래·위 끝은 Q1·Q3이고, 수염은 보통 IQR 경계 안쪽의 가장 먼 관측값까지 뻗습니다. 수염 밖의 점은 오류로 확정된 값이 아니라 원본과 업무 맥락을 확인할 후보입니다.


In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))
df.boxplot(column='이용시간(분)', ax=ax)
ax.set_title('이용시간(분) 분포')
plt.show()


**그래프 ⑥ 히스토그램 — 이용거리 분포**

질문: 이용거리는 어떤 모양으로 분포합니까? → boxplot이 이상치 후보를 강조한다면, 히스토그램은 분포의 전체적인 모양을 보여 줍니다. 분포의 본체를 자세히 보기 위해 상위 1%를 제외하며, 제목에 제외 기준을 함께 표시합니다.


In [ ]:
cutoff = df['이용거리(M)'].quantile(0.99)
fig, ax = plt.subplots(figsize=(7, 4))
df.loc[df['이용거리(M)'] <= cutoff, '이용거리(M)'].plot(kind='hist', bins=40, ax=ax)
ax.set_xlabel('이용거리(M)')
ax.set_title('이용거리 분포 (상위 1% 제외)')
plt.show()


### 인사이트 정리 (예시)

그래프 하나마다 다음 세 질문에 답하면 관찰과 해석을 차근차근 연결할 수 있습니다. **① 무엇이 궁금해서 그렸습니까? → ② 무엇을 관찰했습니까? → ③ 어떤 의미로 해석할 수 있습니까?**

- 시간대별 그래프에서 8시와 18시 전후의 뚜렷한 피크를 관찰했습니다. 이는 통근 이용 가설과 잘 맞지만, 이용 목적 자료로 추가 확인할 필요가 있습니다.
- 히트맵에서는 평일의 출퇴근 피크와 주말 오후의 완만한 분산이 대비됩니다. 평일과 주말의 이용 목적이 다를 가능성을 보여 주며, 대여소 위치나 이용 목적 자료로 확인할 수 있습니다.
- 이용시간 boxplot에서는 대부분이 20분 이내이고 일부가 수백~수천 분까지 이어집니다. 반납 누락이나 거치대 오류뿐 아니라 실제 장시간 이용 가능성도 함께 살펴봅니다.
- 나이에 음수와 비현실적으로 큰 값이 있으므로 계산식과 원본 `생년`을 차례로 확인합니다. 그 결과를 바탕으로 수정·결측 처리·제외 가운데 적절한 방법을 선택합니다.


## Part 3. KRX 정보데이터시스템 API 실습

이 파트에서는 **KRX 지수 1일 단면**으로 API 원리를 익힌 뒤, **코스닥 종목 1일 단면**을 과제 데이터로 저장합니다. 인증키 값은 코드나 출력에 넣지 않고 프로젝트 루트의 `.env`에 있는 `KRX_API_KEY`에서 읽습니다. HTTP 요청 헤더 이름만 KRX 명세에 맞게 `AUTH_KEY`를 사용합니다.

> 인증키 노출과 원본 재배포를 방지하기 위해 `dataset/`과 `.env`는 Git에서 제외됩니다. KRX 데이터는 각 학습자가 승인받은 키로 직접 수집해 로컬에 보관합니다.


### 0. API가 무엇인가 — 개념부터

**API(Application Programming Interface)** 는 '이런 형식으로 요청하면 이런 형식으로 데이터를 주겠다'는 약속입니다. 우리가 웹사이트에서 데이터를 눈으로 보는 대신, 프로그램이 데이터를 바로 받아갈 수 있게 정해둔 창구라고 생각하면 됩니다.

- **요청(Request)**: '이 주소로, 이런 조건으로 데이터를 달라'고 보내는 것 (오늘은 `requests.get()`)
- **인증키**: 승인된 사용자의 요청임을 확인하고 호출량을 관리하기 위한 값
- **응답(Response)**: 서버가 돌려주는 결과. 대부분 **JSON**이라는 형식으로 옵니다.
- **JSON**: `{"이름": "김민준", "나이": 25}`처럼 키-값 쌍으로 이루어진 텍스트 형식. 파이썬의 딕셔너리(`dict`)와 구조가 거의 같아서, `response.json()` 한 줄이면 파이썬 객체로 바로 바뀝니다.


In [ ]:
# JSON이 파이썬 딕셔너리와 얼마나 비슷한지 미리 확인해봅니다 (네트워크 요청 없이)
import json

example_json_text = '{"종목명": "삼성전자", "종가": 71000, "등락률": -0.5}'
parsed = json.loads(example_json_text)
print(type(parsed), parsed)
print('종가만 꺼내기:', parsed['종가'])


이렇게 JSON 텍스트를 파이썬 딕셔너리로 바꾸고 나면, 평소 하던 대로 `pd.DataFrame(...)`으로 표를 만들 수 있습니다. 실제 KRX 응답도 구조만 더 복잡할 뿐 원리는 같습니다.


### 1. 인증키와 API 활용 승인

1. [KRX OPEN API 공식 이용방법](https://openapi.krx.co.kr/contents/OPP/INFO/OPPINFO003.jsp)에 따라 로그인하고 인증키를 신청합니다.
2. `KRX 시리즈 일별시세정보`와 `코스닥 일별매매정보`를 각각 활용 신청합니다. 인증키 승인과 서비스 활용 승인은 별도 단계입니다.
3. `.env.example`을 복사해 `.env`를 만들고 `KRX_API_KEY=발급값`으로 설정합니다. 따옴표는 없어도 됩니다.
4. 인증키 노출을 막기 위해 실제 값은 `.env`에 보관하고, 출력·화면 캡처·Git 커밋에는 포함하지 않습니다.


### 2. 두 엔드포인트와 분석 단위

| 역할 | 데이터셋 이름 | 엔드포인트 | 한 행의 단위 |
|---|---|---|---|
| 본문 실습 | `krx_index` | `idx/krx_dd_trd` | 요청일의 KRX 지수 시리즈 하나 |
| 과제 입력 | `kosdaq_stocks` | `sto/ksq_bydd_trd` | 요청일의 코스닥 종목 하나 |

두 API 모두 `basDd=YYYYMMDD` 하나만 받습니다. 따라서 '일별 API'는 한 번에 긴 시계열을 주는 것이 아니라 **한 거래일의 시장 전체 단면**을 줍니다. 여러 날을 분석할 때는 거래일별 응답을 모읍니다.


In [ ]:
from pathlib import Path
import sys

# 노트북을 프로젝트 루트나 01주차 폴더 어디에서 열어도 모듈을 찾습니다.
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [start, *start.parents] if (path / '01주차' / 'krx_api.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트에서 노트북을 실행합니다.')

week_dir = PROJECT_ROOT / '01주차'
if str(week_dir) not in sys.path:
    sys.path.insert(0, str(week_dir))

from krx_api import collect_dataset, load_dataset

BAS_DD = '20240823'  # 실제 거래가 있었던 금요일로 결과를 재현합니다.
for dataset_name in ['krx_index', 'kosdaq_stocks']:
    result = collect_dataset(dataset_name, BAS_DD, root=PROJECT_ROOT)
    state = '캐시' if result['cached'] else 'API'
    print(f"[{state}] {result['label']}: {result['rows']:,}행")


`collect_dataset()`은 내부에서 HTTPS `GET` 요청을 보내고 다음을 검증합니다.

- 환경 변수 `KRX_API_KEY`를 요청 헤더 `AUTH_KEY`로 전달하고, 로그에는 인증키 대신 요청 상태만 남깁니다.
- `timeout`, HTTP 상태, JSON 형식, `OutBlock_1` 목록, 필수 열, 요청일과 응답일, 고유키 중복을 확인합니다.
- 숫자처럼 보이는 문자열의 쉼표를 제거하고 `'-'`는 결측치로 변환합니다.
- 원본 JSON·분석용 CSV·수집 메타데이터를 날짜별로 저장하며, 파일이 있으면 캐시를 재사용합니다.


In [ ]:
krx_index = load_dataset('krx_index', BAS_DD, root=PROJECT_ROOT)
kosdaq = load_dataset('kosdaq_stocks', BAS_DD, root=PROJECT_ROOT)

print('KRX 지수:', krx_index.shape)
print('코스닥 종목:', kosdaq.shape)
print('KRX 지수 열:', krx_index.columns.tolist())
print('코스닥 열:', kosdaq.columns.tolist())


### 3. 저장하기 전에 데이터 계약 검증하기

지수의 후보 고유키는 `BAS_DD + IDX_CLSS + IDX_NM`, 코스닥의 후보 고유키는 `BAS_DD + ISU_CD`입니다. HTTP 200만 확인하고 저장하면 인증 오류 본문이나 예상 밖 스키마를 정상 데이터로 오해할 수 있으므로, 행 수·날짜·키·결측·업무 규칙을 함께 확인합니다.


In [ ]:
index_duplicates = krx_index.duplicated(['BAS_DD', 'IDX_CLSS', 'IDX_NM']).sum()
stock_duplicates = kosdaq.duplicated(['BAS_DD', 'ISU_CD']).sum()
zero_volume = kosdaq['ACC_TRDVOL'].eq(0)
traded = kosdaq.loc[~zero_volume]
ohlc_violations = (
    (traded['TDD_HGPRC'] < traded[['TDD_OPNPRC', 'TDD_CLSPRC']].max(axis=1))
    | (traded['TDD_LWPRC'] > traded[['TDD_OPNPRC', 'TDD_CLSPRC']].min(axis=1))
    | (traded['ACC_TRDVOL'] < 0)
).sum()

print('KRX 지수 고유키 중복:', index_duplicates)
print('코스닥 고유키 중복:', stock_duplicates)
print('KRX 지수 결측 칸:', int(krx_index.isna().sum().sum()))
print('코스닥 결측 칸:', int(kosdaq.isna().sum().sum()))
print('코스닥 무거래 종목:', int(zero_volume.sum()))
print('거래 발생 종목의 OHLC·거래량 규칙 위반:', int(ohlc_violations))


무거래 종목은 종가는 있지만 시가·고가·저가·거래량이 0일 수 있습니다. 따라서 전체 행에 `고가 ≥ 종가` 규칙을 바로 적용하면 정상적인 무거래 종목까지 오류로 표시합니다. 먼저 거래량이 0인지 분리한 뒤 거래가 발생한 행에 OHLC 규칙을 적용합니다.


### 4. 공식 필드명을 읽기 쉬운 이름으로 바꾸기

원본 필드명은 추적 가능성을 위해 CSV에 보존하고, 분석용 복사본에서만 이름을 바꿉니다.


In [ ]:
index_columns_ko = {
    'BAS_DD': '기준일', 'IDX_CLSS': '계열구분', 'IDX_NM': '지수명',
    'CLSPRC_IDX': '종가', 'CMPPREVDD_IDX': '대비', 'FLUC_RT': '등락률',
    'OPNPRC_IDX': '시가', 'HGPRC_IDX': '고가', 'LWPRC_IDX': '저가',
    'ACC_TRDVOL': '거래량', 'ACC_TRDVAL': '거래대금', 'MKTCAP': '시가총액',
}
stock_columns_ko = {
    'BAS_DD': '기준일', 'ISU_CD': '종목코드', 'ISU_NM': '종목명',
    'MKT_NM': '시장', 'SECT_TP_NM': '소속부', 'TDD_CLSPRC': '종가',
    'CMPPREVDD_PRC': '대비', 'FLUC_RT': '등락률', 'TDD_OPNPRC': '시가',
    'TDD_HGPRC': '고가', 'TDD_LWPRC': '저가', 'ACC_TRDVOL': '거래량',
    'ACC_TRDVAL': '거래대금', 'MKTCAP': '시가총액', 'LIST_SHRS': '상장주식수',
}
index_view = krx_index.rename(columns=index_columns_ko)
stock_view = kosdaq.rename(columns=stock_columns_ko)
display(index_view.head(3))
display(stock_view.head(3))


### 5. 첫 EDA — 결측, 등락률, 거래대금

세 그래프를 보기 전에 결과를 예상해 봅니다. 어떤 지수 필드에 결측이 있습니까? 코스닥 등락률은 어느 구간에 가장 많이 모입니까? 거래대금 상위 종목만으로 전체 시장을 설명할 수 있습니까?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

index_missing = krx_index.isna().sum()
index_missing[index_missing > 0].plot(kind='bar', ax=axes[0], color='#5B8FF9')
axes[0].set_title('KRX 지수: 필드별 결측 개수')
axes[0].set_ylabel('결측 행 수')
axes[0].tick_params(axis='x', rotation=45)

kosdaq['FLUC_RT'].plot(kind='hist', bins=40, ax=axes[1], color='#61DDAA')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('코스닥 종목 등락률 분포')
axes[1].set_xlabel('등락률(%)')

top_value = kosdaq.nlargest(10, 'ACC_TRDVAL').sort_values('ACC_TRDVAL')
axes[2].barh(top_value['ISU_NM'], top_value['ACC_TRDVAL'] / 1e8, color='#F6BD16')
axes[2].set_title('거래대금 상위 10개 종목')
axes[2].set_xlabel('거래대금(억원)')

fig.suptitle(f'KRX OPEN API 스냅샷 — {BAS_DD}')
fig.text(0.99, 0.01, '출처: 한국거래소 통계정보', ha='right', fontsize=9)
plt.tight_layout()
plt.show()


### 실행 후 체크리스트

- [ ] 두 응답의 행 수와 열 이름을 `notion.md`의 대표 결과와 비교했습니다.
- [ ] `OutBlock_1` 누락과 휴장일의 빈 목록이 서로 다른 상태임을 설명할 수 있습니다.
- [ ] 지수 행과 종목 행을 날짜만으로 바로 조인하면 다대다 결합이 되는 이유를 설명할 수 있습니다.
- [ ] 무거래 종목을 분리한 뒤 OHLC 논리 규칙을 검사했습니다.
- [ ] 그래프마다 질문·관찰·해석·한계를 한 문장씩 적었습니다.
- [ ] 다른 거래일로 바꿀 때 기존 파일을 덮어쓰지 않고 날짜별 캐시가 생기는지 확인했습니다.
